# SpAM design simulation (task-v5, stage 2)

Reproduces every number and figure in `report_v5.html`. The HTML page is the shareable
artifact; this notebook is how to check it, poke at it, or re-cut any of it.

**Nothing here recomputes the simulation.** It reads the CSVs a finished run produced. To
rebuild those, see `Cookbook.md` (stage 2 on EC2, then `run_cluster_analysis` locally).

| | |
|---|---|
| Run | `sim_results/design-comparison-v5` |
| Ground truth | `gt_pre_shine_d8.npy` (D=8; the pilot scan selected D=3) |
| Grid | 4 N x 4 screening x 2 arms x 3 softness x 3 dispersion x 6 ndims x 10 reps |
| Fits | 17,280 after de-duplication |

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.io as pio

sys.path.insert(0, str(Path.cwd().parent))   # run from the repo root or from SpAM_Simulations/
pio.renderers.default = "notebook"
pd.set_option("display.width", 200, "display.max_columns", 40)

from SpAM_Simulations.build_report import Run
from SpAM_Simulations import report_clusters as rc, report_sections as rs

RUN = Path("sim_results/design-comparison-v5")
run = Run(RUN)
print(f"{len(run.tables)} tables loaded:")
for name in sorted(run.tables):
    print(f"  {name:<32} {run.tables[name].shape}")
run.calibration

## 1. The design comparison

The headline result. Both arms collect identical numbers of judgements; they differ only in
which image pairs receive them.

In [ ]:
cov = run.get("coverage")
summary = cov.groupby(["num_subjects", "arm"])[
    ["pair_coverage", "average_pair_obs", "median_test_retest", "screening_pass_rate"]
].mean().round(3)
gain = cov.groupby(["num_subjects", "arm"])["pair_coverage"].mean().unstack()
gain["relative gain"] = (100 * (gain["designed"] - gain["random"]) / gain["random"]).round(1)
display(summary, gain.round(2))

### The trade the design makes

Coverage is bought with per-pair precision. At fixed effort, spreading judgements over more
pairs means fewer observations each, so the pre-MDS reliability of the designed arm is *lower*
through the deployable range. This is the same fact as the coverage gain, seen from the other
side - not a contradiction of it.

In [ ]:
stab = run.get("stability")
trade = pd.DataFrame({
    "pair coverage (%)": cov.groupby(["num_subjects", "arm"])["pair_coverage"].mean(),
    "observations per pair": cov.groupby(["num_subjects", "arm"])["average_pair_obs"].mean(),
    "pre-MDS Spearman": stab.groupby(["num_subjects", "arm"])["spearman"].mean(),
}).round(3)
display(trade)
rs.section_coverage(run)  # the report's own figures, rendered inline
None

## 2. Granularity: the central negative result

Two independent cohorts are clustered at every `k` and asked whether they find the same groups.
`k*` is selected per configuration by two rules built to disagree where real structure exists -
VI rewards parsimony, silhouette rewards separation.

In [ ]:
ks = run.get("k_selection")
print(f"configurations: {len(ks):,}")
print("k_star_vi :", ks["k_star_vi"].value_counts().sort_index().to_dict())
print("k_star_sil:", ks["k_star_sil"].value_counts().sort_index().to_dict())

ag = run.get("cluster_agreement")
curve = ag.groupby(["linkage", "k"])[["mean_vi_norm", "mean_sil_cross", "mean_ari"]].mean()
display(curve.round(3).unstack(0))

The cross-cohort silhouette crossing zero is the decisive number: beyond roughly `k=12`, a
cohort's clusters are *not* separated in another cohort's geometry, so what the algorithm returns
there is a slicing of a continuum rather than a set of groups.

In [ ]:
zero_crossing = curve.reset_index().query("mean_sil_cross < 0").groupby("linkage")["k"].min()
print("first k with negative cross-cohort silhouette, per linkage:")
print(zero_crossing.to_string())

## 3. The ground truth is the binding constraint

Run `python -m SpAM_Simulations.gt_diagnostics` to regenerate these. `frac_of_ceiling` above 1
means the embedding reproduces variance the raw data cannot reproduce in itself - it is fitting
noise, and the apparent within-level agreement is an artefact.

In [ ]:
gt_raw, ceiling = run.get("gt_vs_raw"), run.get("noise_ceiling")
if gt_raw is not None:
    display(gt_raw[["level_name", "n_observed", "spearman", "ceiling_full",
                    "frac_of_ceiling"]].round(3))
else:
    print("run gt_diagnostics first")

## 4. Validity: the check nothing was fitted to

In the pilot, the disagreement between a participant's two judgements of the same pair is an
inverted U against how far apart they placed it. The high-distance turnover requires a bounded
canvas, and the model was never fitted to it.

In [ ]:
display(run.get("noise_curve_shape").round(3))
rc.section_validity(run)
None

## 5. Rebuild the shareable report

In [ ]:
from SpAM_Simulations.build_report import build

out = RUN / "report_v5.html"
out.write_text(build(run), encoding="utf-8")
print(f"wrote {out} ({out.stat().st_size / 1e6:.1f} MB)")